# OpenPlaque — RCA Expert Outer-Wall Validation v1

Consumes a blinded expert outer-wall mask and compares it with the locked RCA plaque proxy **without retuning**. Use **Runtime → Run all**.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, os, shutil, sys, time
DRIVE_ROOT = Path('/content/drive/MyDrive/OpenPlaque')
OUTPUT = DRIVE_ROOT / 'RCA_Expert_Outer_Wall_Validation_v1'
REUSE_VALID_CACHES = True
FORCE_RECOMPUTE = False
if FORCE_RECOMPUTE and OUTPUT.exists(): shutil.rmtree(OUTPUT)
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT/'notebook_started.json').write_text(json.dumps({'status':'started','time':time.time()}, indent=2))
print('Drive root:', DRIVE_ROOT)
print('Output:', OUTPUT)


In [ ]:
import os, shutil, sys
os.chdir('/content')
REPO = Path('/content/OpenPlaque_rca_expert_wall_validation')
if REPO.exists(): shutil.rmtree(REPO)
BRANCH = 'rca-expert-outer-wall-validation-from-main'
PINNED_SCIENCE_COMMIT = '5c7f2ae5925a5bdc885d47666863ccea32c4d2fd'
BASELINE = '0593b453959f5a353d644267fbeef24b514ef4d7'
print('Working directory repaired:', os.getcwd())
!git clone -q --branch $BRANCH https://github.com/pazzani/OpenPlaque.git $REPO
!git -C $REPO checkout -q $PINNED_SCIENCE_COMMIT
HEAD = get_ipython().getoutput(f'git -C {REPO} rev-parse HEAD')[0].strip()
MB = get_ipython().getoutput(f'git -C {REPO} merge-base HEAD {BASELINE}')[0].strip()
print('Checked out:', HEAD)
print('Merge base:', MB)
assert HEAD == PINNED_SCIENCE_COMMIT
assert MB == BASELINE
%pip install -q /content/OpenPlaque_rca_expert_wall_validation
for k in list(sys.modules):
    if k == 'openplaque' or k.startswith('openplaque.'):
        del sys.modules[k]
os.chdir('/content')


In [ ]:
from openplaque.rca_expert_outer_wall_validation_v1 import synthetic_self_test
print('Synthetic self-test:', synthetic_self_test())
!pytest -q /content/OpenPlaque_rca_expert_wall_validation/tests/test_rca_expert_outer_wall_validation_v1.py


In [ ]:
required = [
    DRIVE_ROOT/'Cache/Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json',
    DRIVE_ROOT/'RCA_Expert_Outer_Wall_Annotation_Pack_v1/expert_pack/RCA_expert_outer_wall_planes.npz',
    DRIVE_ROOT/'RCA_Expert_Outer_Wall_Annotation_Pack_v1/expert_pack/RCA_expert_outer_wall_manifest.csv',
    DRIVE_ROOT/'RCA_Plaque_PCAT_Research_Lock_v1/summary.json',
    DRIVE_ROOT/'RCA_Plaque_PCAT_Research_Lock_v1/RCA_locked_research_plaque_profile_1mm.csv',
]
missing = [str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError('Missing prerequisites:\n' + '\n'.join(missing))
(OUTPUT/'preflight_complete.json').write_text(json.dumps({'status':'complete','science_commit':PINNED_SCIENCE_COMMIT,'baseline':BASELINE,'required_count':len(required)}, indent=2))
print('Preflight complete:', len(required), 'required artifacts found')


In [ ]:
from openplaque.rca_expert_outer_wall_validation_v1 import run
result = run(drive_root=str(DRIVE_ROOT), output_dir=str(OUTPUT))
print(json.dumps(result['summary'], indent=2, default=str))
if result['summary']['status'] == 'RCA_EXPERT_OUTER_WALL_MASK_MISSING':
    print('\nExpert mask not found yet. Place RCA_outer_wall_expert_mask.npy in the expert_pack folder, annotation-pack folder, or OpenPlaque root and rerun.')
else:
    print('Report:', result.get('report'))
    print('ZIP:', result.get('zip'))
